# ND Week 3 — Forward/Backward Selection, PCR, and PLSR
**DX799 Capstone | Cindy Lopez**

**Question:** Can feature selection or lower-dimensional components improve out-of-sample prediction of marketing outcomes across three datasets? Each dataset is modeled separately, with `ConversionRate` for customer-level records and `Conversions` for campaign-level records. No cross-dataset RMSE comparisons are valid because outcome units differ.

In [6]:
import numpy as np, pandas as pd, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display

FILES = {
 'Customer conversion': 'digital_marketing_campaign_dataset.csv',
 'Google Ads': 'GoogleAds_DataAnalytics_Sales_Uncleaned.csv',
 'Product performance': 'marketing_and_product_performance.csv'
}
for label, filename in FILES.items():
    if not os.path.isfile(filename):
        raise FileNotFoundError(f'Missing {filename}.')
print('All three CSV files found.')

All three CSV files found.


## 1. Dataset EDA and cleaning
Inspect missingness and distributions before modeling. Numeric missing values are imputed on training data only; categorical variables are encoded without ordinal assumptions. Exclude target proxies and downstream outcome variables.

In [2]:
data={name:pd.read_csv(filename) for name,filename in FILES.items()}
for name, df in data.items():
    print('\n',name,'shape:',df.shape,'missing cells:',int(df.isna().sum().sum()),'duplicates:',int(df.duplicated().sum()))
    display(df.head(2))
    display(df.isna().sum().sort_values(ascending=False).head(5).to_frame('missing'))


 Customer conversion shape: (8000, 20) missing cells: 0 duplicates: 0


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1


,missing
CustomerID,0
Age,0
AdvertisingTool,0
AdvertisingPlatform,0
LoyaltyPoints,0



 Google Ads shape: (2600, 13) missing cells: 1150 duplicates: 0


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0,4498.0,$231.88,14.0,7.0,0.058,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0,5107.0,$216.84,10.0,8.0,0.046,$1679,20-11-2024,hyderabad,mobile,data analytics course


,missing
Conversion Rate,626
Sale_Amount,139
Clicks,112
Cost,97
Conversions,74



 Product performance shape: (10000, 17) missing cells: 0 duplicates: 0


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative


,missing
Campaign_ID,0
Subscription_Length,0
Customer_Satisfaction_Post_Refund,0
Bundle_Price,0
Bundle_ID,0


In [3]:
ads=data['Google Ads'].copy()
for col in ['Cost','Sale_Amount']:
    ads[col]=pd.to_numeric(ads[col].astype(str).str.replace(r'[$,]','',regex=True),errors='coerce')
for col in ['Clicks','Impressions','Leads','Conversions','Conversion Rate']:
    ads[col]=pd.to_numeric(ads[col],errors='coerce')
# We deliberately exclude Leads, Conversion Rate and Sale_Amount as outcome-adjacent or downstream variables.
config={
 'Customer conversion':{'frame':data['Customer conversion'],'target':'ConversionRate','numeric':['Age','Income','AdSpend','ClickThroughRate','WebsiteVisits','PagesPerVisit','TimeOnSite','SocialShares','EmailOpens','EmailClicks','PreviousPurchases','LoyaltyPoints'],'categorical':['Gender','CampaignChannel','CampaignType'],'poly':['AdSpend','ClickThroughRate','TimeOnSite']},
 'Google Ads':{'frame':ads,'target':'Conversions','numeric':['Clicks','Impressions','Cost'],'categorical':['Campaign_Name','Location','Device'],'poly':['Clicks','Impressions','Cost']},
 'Product performance':{'frame':data['Product performance'],'target':'Conversions','numeric':['Budget','Clicks','Discount_Level','Bundle_Price'],'categorical':['Subscription_Tier','Common_Keywords'],'poly':['Budget','Clicks','Discount_Level']}
}
for name,c in config.items():
    print(name,'target:',c['target'],'range:',c['frame'][c['target']].min(),c['frame'][c['target']].max())

Customer conversion target: ConversionRate range: 0.0100177828330717 0.1999947079999166
Google Ads target: Conversions range: 3.0 10.0
Product performance target: Conversions range: 1 999


## 2. Feature selection, PCR, and PLSR !!!!
Use an 80/20 holdout split. Forward and backward sequential selection choose a subset of *encoded* features based on training-fold cross-validation. PCR applies PCA to standardized encoded features, then fits linear regression on the selected components. PLSR constructs components that consider both predictor variation and the outcome. Tune the number of features/components with training-only five-fold cross-validation. 

In [4]:
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline

def make_encoder():
    try:return OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError:return OneHotEncoder(handle_unknown='ignore',sparse=False)
cv=KFold(n_splits=3,shuffle=True,random_state=42)
rows=[]
for name,c in config.items():
    cols=c['numeric']+c['categorical']; target=c['target']; work=c['frame'][cols+[target]].copy()
    work[target]=pd.to_numeric(work[target],errors='coerce'); work=work.dropna(subset=[target])
    xtr,xte,ytr,yte=train_test_split(work[cols],work[target],test_size=.2,random_state=42)
    prep=ColumnTransformer([
      ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),c['numeric']),
      ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',make_encoder())]),c['categorical'])])
    Xtr=prep.fit_transform(xtr); Xte=prep.transform(xte)
    # Standardize all encoded features for PCR/PLSR; fitted only on training observations
    scaler=StandardScaler(); A=scaler.fit_transform(Xtr); B=scaler.transform(Xte)
    k=min(5,A.shape[1]-1)
    models={'Linear baseline':LinearRegression()}
    for direction in ('forward','backward'):
        selector=SequentialFeatureSelector(LinearRegression(),n_features_to_select=k,direction=direction,cv=3,scoring='neg_mean_squared_error',n_jobs=-1)
        # Restrict selection fit to a reproducible subsample for tractability
        rng=np.random.default_rng(42); inds=rng.choice(len(A),size=min(750,len(A)),replace=False)
        selector.fit(A[inds],np.asarray(ytr)[inds])
        reg=LinearRegression().fit(selector.transform(A),ytr)
        pred=reg.predict(selector.transform(B))
        rows.append({'dataset':name,'model':direction+' selection','test_R2':r2_score(yte,pred),'test_RMSE':np.sqrt(mean_squared_error(yte,pred)),'test_MAE':mean_absolute_error(yte,pred),'selected_n':k,'components':np.nan})
    baseline=LinearRegression().fit(A,ytr)
    pred=baseline.predict(B)
    rows.append({'dataset':name,'model':'Linear baseline','test_R2':r2_score(yte,pred),'test_RMSE':np.sqrt(mean_squared_error(yte,pred)),'test_MAE':mean_absolute_error(yte,pred),'selected_n':np.nan,'components':np.nan})
    for label,est,parameter in [('PCR',Pipeline([('pca',PCA()),('reg',LinearRegression())]),'pca__n_components'),('PLSR',PLSRegression(scale=False),'n_components')]:
        candidates=sorted(set([1,2,3,5,8,min(12,A.shape[1]-1)]))
        candidates=[v for v in candidates if v<=A.shape[1]]
        gs=GridSearchCV(est,{parameter:candidates},cv=cv,scoring='neg_root_mean_squared_error',n_jobs=-1)
        gs.fit(A,ytr); pred=np.asarray(gs.predict(B)).ravel()
        rows.append({'dataset':name,'model':label,'test_R2':r2_score(yte,pred),'test_RMSE':np.sqrt(mean_squared_error(yte,pred)),'test_MAE':mean_absolute_error(yte,pred),'selected_n':np.nan,'components':gs.best_params_[parameter]})
    print('Completed',name,'encoded feature count:',A.shape[1])
results=pd.DataFrame(rows)
display(results.round(4))

Completed Customer conversion encoded feature count: 23
Completed Google Ads encoded feature count: 20
Completed Product performance encoded feature count: 11


,dataset,model,test_R2,test_RMSE,test_MAE,selected_n,components
0,Customer conversion,forward selection,-0.0003,0.0540,0.0469,5.0,NaN
1,Customer conversion,backward selection,-0.0000,0.0540,0.0469,5.0,NaN
2,Customer conversion,Linear baseline,-0.0023,0.0541,0.0469,NaN,NaN
3,Customer conversion,PCR,-0.0017,0.0541,0.0469,NaN,1.0
4,Customer conversion,PLSR,-0.0024,0.0541,0.0469,NaN,2.0
5,Google Ads,forward selection,-0.0035,2.2594,1.9618,5.0,NaN
6,Google Ads,backward selection,-0.0056,2.2618,1.9640,5.0,NaN
7,Google Ads,Linear baseline,0.0007,2.2547,1.9564,NaN,NaN
8,Google Ads,PCR,-0.0002,2.2558,1.9626,NaN,1.0
9,Google Ads,PLSR,0.0007,2.2547,1.9564,NaN,2.0


## 3. Compare performance and interpret components
The lowest test RMSE within a dataset indicates the best observed holdout performance, but differences can be small and sample-dependent. PCR prioritizes directions with high predictor variance; PLSR also uses covariance with the target. Sequential selection is greedy, not a guarantee of the globally best subset. The selection step uses a capped training subsample and three-fold CV for computational feasibility, which can increase instability. The test set remains held out from selection and tuning.

In [5]:
display(results.loc[results.groupby('dataset')['test_RMSE'].idxmin(),['dataset','model','test_R2','test_RMSE','components','selected_n']].round(4))

,dataset,model,test_R2,test_RMSE,components,selected_n
1,Customer conversion,backward selection,-0.0000,0.0540,NaN,5.0
9,Google Ads,PLSR,0.0007,2.2547,2.0,NaN
14,Product performance,PLSR,-0.0006,291.0499,2.0,NaN


In [7]:

# Enhanced Week 3 analysis:
# Compare all models against the linear baseline

week3_comparison = results.copy()

comparison_rows = []

for dataset_name, group in week3_comparison.groupby("dataset"):

    baseline_rmse = group.loc[
        group["model"] == "Linear baseline",
        "test_RMSE"
    ].iloc[0]

    for _, row in group.iterrows():

        change = (
            (row["test_RMSE"] - baseline_rmse)
            / baseline_rmse
        ) * 100

        comparison_rows.append({
            "Dataset": dataset_name,
            "Model": row["model"],
            "Test R2": row["test_R2"],
            "Test RMSE": row["test_RMSE"],
            "RMSE Change (%)": change,
            "Selected Features": row["selected_n"],
            "Components": row["components"]
        })

week3_summary = pd.DataFrame(comparison_rows)

for dataset_name, group in week3_summary.groupby("Dataset"):
    print(f"\n{dataset_name}")
    display(group.round(5))



Customer conversion


,Dataset,Model,Test R2,Test RMSE,RMSE Change (%),Selected Features,Components
0,Customer conversion,forward selection,-0.00029,0.05404,-0.09936,5.0,NaN
1,Customer conversion,backward selection,-0.00001,0.05403,-0.11306,5.0,NaN
2,Customer conversion,Linear baseline,-0.00228,0.05409,0.00000,NaN,NaN
3,Customer conversion,PCR,-0.00168,0.05408,-0.02989,NaN,1.0
4,Customer conversion,PLSR,-0.00236,0.05410,0.00409,NaN,2.0



Google Ads


,Dataset,Model,Test R2,Test RMSE,RMSE Change (%),Selected Features,Components
5,Google Ads,forward selection,-0.00348,2.25944,0.20884,5.0,NaN
6,Google Ads,backward selection,-0.00558,2.26181,0.31381,5.0,NaN
7,Google Ads,Linear baseline,0.00070,2.25473,0.00000,NaN,NaN
8,Google Ads,PCR,-0.00025,2.25580,0.04737,NaN,1.0
9,Google Ads,PLSR,0.00073,2.25470,-0.00142,NaN,2.0



Product performance


,Dataset,Model,Test R2,Test RMSE,RMSE Change (%),Selected Features,Components
10,Product performance,forward selection,-0.00245,291.31684,0.08998,5.0,NaN
11,Product performance,backward selection,-0.00141,291.16578,0.03808,5.0,NaN
12,Product performance,Linear baseline,-0.00065,291.05495,0.00000,NaN,NaN
13,Product performance,PCR,-0.00073,291.06595,0.00378,NaN,5.0
14,Product performance,PLSR,-0.00062,291.04993,-0.00173,NaN,2.0


In [8]:

# Compact Week 3 comparison table
# Focus on performance and model complexity

compact_week3 = week3_summary[
    [
        "Dataset",
        "Model",
        "Test R2",
        "Test RMSE",
        "RMSE Change (%)",
        "Selected Features",
        "Components"
    ]
].copy()

# Shorten the column headings
compact_week3 = compact_week3.rename(columns={
    "Dataset": "Dataset",
    "Model": "Model",
    "Test R2": "R2",
    "Test RMSE": "RMSE",
    "RMSE Change (%)": "RMSE Change %",
    "Selected Features": "Features",
    "Components": "PCs"
})

for dataset_name, group in compact_week3.groupby("Dataset"):

    print(f"\n{dataset_name}")

    display(
        group.drop(columns="Dataset")
        .round(5)
        .to_string(index=False)
    )



Customer conversion


'             Model       R2    RMSE  RMSE Change %  Features  PCs\n forward selection -0.00029 0.05404       -0.09936       5.0  NaN\nbackward selection -0.00001 0.05403       -0.11306       5.0  NaN\n   Linear baseline -0.00228 0.05409        0.00000       NaN  NaN\n               PCR -0.00168 0.05408       -0.02989       NaN  1.0\n              PLSR -0.00236 0.05410        0.00409       NaN  2.0'


Google Ads


'             Model       R2    RMSE  RMSE Change %  Features  PCs\n forward selection -0.00348 2.25944        0.20884       5.0  NaN\nbackward selection -0.00558 2.26181        0.31381       5.0  NaN\n   Linear baseline  0.00070 2.25473        0.00000       NaN  NaN\n               PCR -0.00025 2.25580        0.04737       NaN  1.0\n              PLSR  0.00073 2.25470       -0.00142       NaN  2.0'


Product performance


'             Model       R2      RMSE  RMSE Change %  Features  PCs\n forward selection -0.00245 291.31684        0.08998       5.0  NaN\nbackward selection -0.00141 291.16578        0.03808       5.0  NaN\n   Linear baseline -0.00065 291.05495        0.00000       NaN  NaN\n               PCR -0.00073 291.06595        0.00378       NaN  5.0\n              PLSR -0.00062 291.04993       -0.00173       NaN  2.0'


### Enhanced Interpretation: Feature Selection and Dimensionality Reduction

The Week 3 results demonstrate that reducing the number of predictors or constructing lower-dimensional representations does not consistently improve predictive performance across digital marketing datasets.

For the customer conversion dataset, backward selection achieved the lowest test RMSE of 0.05403, representing a 0.11306% improvement over ordinary linear regression. Forward selection also improved RMSE by 0.09936%. Both methods retained five encoded features, suggesting that a smaller predictor set may slightly improve generalization. However, the best test R² remained approximately zero, indicating that the improvement was limited.

The Google Ads dataset showed a different pattern. Forward and backward selection increased test RMSE by 0.20884% and 0.31381%, respectively. PLSR achieved the lowest RMSE of 2.25470 using two components, but improved on the baseline by only 0.00142%. Its test R² of 0.00073 indicates that the model explained almost none of the observed variation in conversions.

For product performance, PLSR again achieved the lowest RMSE using two components. However, its improvement over linear regression was only 0.00173%, and its test R² remained negative. Forward selection, backward selection, and PCR produced slightly higher prediction errors than the baseline.

These findings highlight an important distinction between feature selection and dimensionality reduction. Forward and backward selection retain subsets of encoded predictors, whereas PCR constructs components based on predictor variance and PLSR constructs components using information about the relationship between predictors and the outcome. Therefore, selecting five features is not directly equivalent to retaining five components.

Overall, the results suggest that simpler model representations can maintain comparable predictive performance, but they do not necessarily improve generalization. The most consistent finding across the three datasets is that the available regression predictors explain very little variation in the target outcomes.

This analysis reinforces the broader project conclusion that model complexity alone is not a reliable indicator of predictive usefulness.



## Enhanced Analysis: Feature Selection, Dimensionality Reduction, and Generalization

### Research Question

Do feature selection and dimensionality reduction meaningfully improve prediction across digital marketing datasets, or do they primarily simplify model representations?

This analysis compares forward selection, backward selection, principal components regression (PCR), and partial least squares regression (PLSR) against ordinary linear regression.

The evaluation focuses on held-out test RMSE, test R², the number of selected features, and the number of retained components.

Because each dataset has a different outcome scale, percentage changes in RMSE are calculated relative to the linear baseline within the same dataset.

The purpose is to evaluate whether additional model-selection complexity is justified by measurable improvements in generalization.



### Methodological Limitations

Forward and backward selection were evaluated using a fixed five-feature subset and a training subsample to reduce computational cost. Consequently, the results do not establish that five features represent the optimal subset size. PCR and PLSR component counts were selected through training-only cross-validation, which reduces the risk of choosing model complexity based on test-set performance. The analysis uses a single held-out test split, so small differences in RMSE should not be interpreted as evidence of statistically reliable improvements without further validation.


## 4. Overfitting and limitations
Training-only preprocessing, selection, and cross-validation limit leakage. Dimensionality reduction can reduce variance, but discarding predictive directions can increase bias. PCR and PLSR component counts were tuned using three-fold CV; forward/backward selection uses three-fold CV on a training subsample and a fixed subset size, so it is exploratory rather than exhaustive. No causal interpretation is warranted, and the three datasets have different measurement units and sampling designs.

## References and AI use
- El Kharoua, R. (2024). *Predict conversion in digital marketing dataset*. Kaggle. https://www.kaggle.com/datasets/rabieelkharoua/predict-conversion-in-digital-marketing-dataset
- Nayak, G. (2024). *Google Ads sales dataset*. Kaggle. https://www.kaggle.com/datasets/nayakganesh007/google-ads-sales-dataset
- Shah, I. A. (2024). *Marketing and product performance dataset*. Kaggle. https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset
- Scikit-learn developers. *Cross decomposition*. https://scikit-learn.org/stable/modules/cross_decomposition.html

**AI use:** 